# Moissonnage contrôlé : de l’URL au corpus source

**Cours :** Corpus et discours de spécialité : concevoir et conduire une enquête  
**Langage :** Python  
**Étape Git :** `chaine/03_acquisition/`

Ce notebook adapte l’ancien document R Markdown de moissonnage à une chaîne Python compatible avec le dépôt du cours.

## Objectifs

À la fin du notebook, l’étudiant pourra :

1. distinguer **source Web**, **document source**, **métadonnées**, **texte extrait** et **corpus préparé** ;
2. inspecter une page Web et identifier un sélecteur CSS ou XPath ;
3. collecter une liste finie d’URL explicitement documentées ;
4. sauvegarder le HTML brut et les métadonnées de provenance ;
5. extraire des métadonnées HTML et un aperçu textuel ;
6. comprendre pourquoi l’extraction PDF, le nettoyage et la tokenisation sont des étapes distinctes ;
7. produire des fichiers CSV et JSON lisibles et versionnables.

> **Principe :** le moissonnage est une modalité d’acquisition parmi d’autres. Une API, un export, un téléchargement manuel ou un corpus existant peuvent être préférables selon la question, les droits et les conditions d’accès.


## Structure Git visée

```text
gabarit_projet/
├── configuration/
│   └── urls_a_collecter.csv
├── donnees/
│   ├── sources/                 # généralement ignoré par Git
│   └── intermediaires/          # généralement ignoré par Git
├── chaine/
│   ├── 03_acquisition/
│   │   ├── code/
│   │   └── logs/
│   └── 04_preparation/
│       └── 01_extraction/
└── journal/
```

Le notebook peut être stocké dans :

```text
chaine/03_acquisition/notebooks/01_moissonnage_exemple.ipynb
```

Les fichiers produits ne deviennent pas automatiquement le corpus final. Ils doivent être vérifiés, documentés dans `donnees/inventaire.csv`, puis préparés dans les étapes suivantes.


## Installation des bibliothèques

Dans l’environnement Python du projet, installer les dépendances suivantes si nécessaire :

```bash
python -m pip install requests beautifulsoup4 pandas pypdf selenium
```

Pour l’inspection facultative dans Chrome avec Selenium :

```bash
python -m pip install "selenium>=4.11,<5"
```

Selenium est facultatif dans ce notebook : l’acquisition contrôlée et l’extraction des métadonnées utilisent principalement `requests`, `BeautifulSoup` et `pandas`. Selenium sert à rendre visible le navigateur, le DOM et les sélecteurs.


In [ ]:
# 1. Importer les bibliothèques
from __future__ import annotations

import csv
import hashlib
import json
import re
import time
import urllib.robotparser
from datetime import UTC, datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

print('Imports réussis.')


## Configuration du projet

Dans un notebook, `__file__` n’est généralement pas défini. Le répertoire de travail doit donc être explicitement configuré sur la racine du projet étudiant : le dossier qui contient directement `chaine/`, `donnees/`, `configuration/` et `journal/`.

Lors d’une première démonstration, le notebook crée les dossiers nécessaires s’ils n’existent pas encore.


In [ ]:
# 2. Définir la racine du projet
#
# Dans Spyder ou Jupyter, exécuter ce notebook depuis le dossier
# gabarit_projet/ ou modifier explicitement la ligne PROJECT_ROOT.

PROJECT_ROOT = Path.cwd().resolve()

# Exemple de configuration explicite sous Windows :
# PROJECT_ROOT = Path('D:/NLP/Cours/Langues_de_specialite/gabarit_projet').resolve()

CONFIG_DIR = PROJECT_ROOT / 'configuration'
SOURCES_DIR = PROJECT_ROOT / 'donnees' / 'sources'
INTERMEDIATE_DIR = PROJECT_ROOT / 'donnees' / 'intermediaires'
ACQUISITION_DIR = PROJECT_ROOT / 'chaine' / '03_acquisition'
LOGS_DIR = ACQUISITION_DIR / 'logs'

for directory in (CONFIG_DIR, SOURCES_DIR, INTERMEDIATE_DIR, ACQUISITION_DIR, LOGS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('Racine du projet :', PROJECT_ROOT)
print('Sources :', SOURCES_DIR)
print('Intermédiaires :', INTERMEDIATE_DIR)


## Conditions d’acquisition

Avant une collecte réelle, vérifier :

- les conditions d’utilisation de la source ;
- les droits de collecte, de réutilisation et de redistribution ;
- l’existence d’une API ou d’un export préférable ;
- la présence éventuelle de données personnelles ;
- les limites de fréquence ;
- les règles techniques de `robots.txt`, lorsqu’elles sont pertinentes ;
- la possibilité de stocker ou partager les documents acquis.

`robots.txt` peut fournir une indication technique sur les chemins demandés aux agents automatisés, mais ne remplace pas une autorisation de réutilisation ou de redistribution.

Le notebook est configuré par défaut pour une **démonstration locale**, sans requête réseau.


In [ ]:
# 3. Paramètres d'exécution
EXECUTER_DEMONSTRATION_LOCALE = True
EXECUTER_COLLECTE_REELLE = False
CONDITIONS_ACCES_VERIFIEES = False

USER_AGENT = 'LanguesDeSpecialiteCours/1.0 (contact: a-completer)'
ROBOT_AGENT = 'LanguesDeSpecialiteCours'
TIMEOUT_SECONDS = 20
DELAI_ENTRE_REQUETES = 2

# Exemple historique conservé depuis le matériel de formation.
URL_EXEMPLE_OPENEDITION = 'https://journals.openedition.org/aad/171'

# Sélecteur à inspecter et adapter pour une source réelle.
SELECTEUR_CONTENU = '#content, article, main'

print('Mode démonstration locale :', EXECUTER_DEMONSTRATION_LOCALE)
print('Collecte réelle :', EXECUTER_COLLECTE_REELLE)


## Construire une liste finie d’URL

Le notebook ne parcourt pas automatiquement tous les volumes d’une revue ni tous les liens d’un site. Les documents à collecter sont explicitement définis dans un fichier CSV versionnable.

Format recommandé :

```csv
id_document,url,genre,commentaire
D0001,https://journals.openedition.org/aad/171,article,Exemple pédagogique OpenEdition
```

Dans un projet réel, ce fichier peut être conservé sous :

```text
configuration/urls_a_collecter.csv
```


In [ ]:
# 4. Créer un fichier de démonstration local
#
# La démonstration crée une page HTML fictive afin que tout le notebook
# puisse être exécuté et vérifié sans dépendre d'Internet.

urls_path = CONFIG_DIR / 'urls_a_collecter.csv'
fixture_path = SOURCES_DIR / 'D0001_source.html'

if EXECUTER_DEMONSTRATION_LOCALE:
    fixture_html = '''<!doctype html>
<html lang="fr">
<head>
  <meta charset="utf-8">
  <meta name="DC.title" content="Exemple local de document spécialisé">
  <meta name="DC.creator" content="Lareau, Francis">
  <meta name="DC.date" content="2026-10-08">
  <meta name="DC.language" content="fr">
  <meta name="citation_journal_title" content="Revue pédagogique fictive">
  <meta name="citation_volume" content="1">
  <meta name="citation_issue" content="1">
  <meta name="citation_abstract" content="Ce document fictif sert à tester le notebook.">
  <meta name="keywords" content="corpus, acquisition, métadonnées">
  <title>Exemple local de document spécialisé</title>
</head>
<body>
  <main id="content">
    <h1>Exemple local de document spécialisé</h1>
    <p>Cette phrase présente un résultat fictif dans un contexte pédagogique.</p>
    <p>Les résultats suggèrent qu'une collecte documentée facilite la vérification.</p>
  </main>
</body>
</html>'''
    fixture_path.write_text(fixture_html, encoding='utf-8')
    pd.DataFrame([{
        'id_document': 'D0001',
        'url': 'https://example.org/document-fictif',
        'genre': 'article',
        'commentaire': 'Fixture locale : aucune collecte réseau.'
    }]).to_csv(urls_path, index=False, encoding='utf-8')

urls_a_collecter = pd.read_csv(urls_path, encoding='utf-8')
urls_a_collecter


## Fonctions utilitaires

Les fonctions suivantes :

- vérifient les champs du CSV ;
- consultent `robots.txt` de manière technique pour une collecte réelle ;
- calculent une empreinte SHA-256 du HTML acquis ;
- extraient des métadonnées de plusieurs conventions HTML ;
- normalisent uniquement les espaces dans l’aperçu textuel.

Les opérations d’extraction stabilisée et de nettoyage approfondi doivent être documentées plus tard dans `04_preparation/`.


In [ ]:
# 5. Fonctions utilitaires

def verifier_liste_urls(dataframe: pd.DataFrame) -> None:
    colonnes_requises = {'id_document', 'url'}
    if not colonnes_requises.issubset(dataframe.columns):
        raise ValueError('Le CSV doit contenir les colonnes id_document et url.')
    if dataframe['id_document'].isna().any() or dataframe['url'].isna().any():
        raise ValueError('Chaque ligne doit contenir id_document et url.')
    if dataframe['id_document'].duplicated().any():
        doublons = dataframe.loc[dataframe['id_document'].duplicated(), 'id_document'].tolist()
        raise ValueError(f'Identifiants dupliqués : {doublons}')


def verifier_robots(url: str, robot_agent: str, timeout_seconds: int) -> bool | None:
    parsed = urlparse(url)
    robots_url = urljoin(f'{parsed.scheme}://{parsed.netloc}', '/robots.txt')
    parser = urllib.robotparser.RobotFileParser()
    parser.set_url(robots_url)
    try:
        parser.read()
        return parser.can_fetch(robot_agent, url)
    except Exception:
        return None


def premier_meta(soup: BeautifulSoup, *noms: str) -> str:
    for nom in noms:
        for attribut in ('name', 'property', 'scheme'):
            balise = soup.find('meta', attrs={attribut: nom})
            if balise and balise.get('content'):
                return balise['content'].strip()
    return ''


def tous_meta(soup: BeautifulSoup, *noms: str) -> list[str]:
    valeurs = []
    for nom in noms:
        for attribut in ('name', 'property', 'scheme'):
            for balise in soup.find_all('meta', attrs={attribut: nom}):
                if balise.get('content'):
                    valeurs.append(balise['content'].strip())
    return list(dict.fromkeys(valeurs))


def normaliser_espaces(texte: str) -> str:
    return re.sub(r'\s+', ' ', texte.replace('\xa0', ' ')).strip()


verifier_liste_urls(urls_a_collecter)
print('Liste d’URL valide.')


## Acquisition HTTP contrôlée

La fonction suivante n’est exécutée que si :

```python
EXECUTER_COLLECTE_REELLE = True
CONDITIONS_ACCES_VERIFIEES = True
```

Elle collecte uniquement les URL présentes dans `urls_a_collecter.csv`, attend entre les requêtes et conserve le HTML brut dans `donnees/sources/`.

Elle ne découvre pas automatiquement de nouvelles pages et ne télécharge pas automatiquement de PDF.


In [ ]:
# 6. Fonction de collecte HTTP contrôlée

def collecter_pages_html(
    documents: pd.DataFrame,
    sources_dir: Path,
    user_agent: str,
    robot_agent: str,
    timeout_seconds: int,
    delai_secondes: int,
) -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({
        'User-Agent': user_agent,
        'Accept': 'text/html,application/xhtml+xml',
    })

    traces = []

    for position, document in documents.reset_index(drop=True).iterrows():
        document_id = str(document['id_document'])
        url = str(document['url'])
        robots_allowed = verifier_robots(url, robot_agent, timeout_seconds)
        timestamp = datetime.now(UTC).isoformat()

        if robots_allowed is not True:
            traces.append({
                'id_document': document_id,
                'url': url,
                'date_collecte_utc': timestamp,
                'robots_allowed': robots_allowed,
                'statut': 'non_collecte',
                'message': 'robots.txt interdit ou ne permet pas de vérifier la collecte.',
            })
            continue

        try:
            response = session.get(url, timeout=timeout_seconds)
            response.raise_for_status()
            content_type = response.headers.get('content-type', '')

            if 'html' not in content_type.lower():
                traces.append({
                    'id_document': document_id,
                    'url': url,
                    'date_collecte_utc': timestamp,
                    'robots_allowed': robots_allowed,
                    'statut': 'format_non_html',
                    'message': content_type,
                })
                continue

            html_path = sources_dir / f'{document_id}_source.html'
            html_path.write_bytes(response.content)
            sha256 = hashlib.sha256(response.content).hexdigest()

            traces.append({
                'id_document': document_id,
                'url': url,
                'url_finale': response.url,
                'date_collecte_utc': timestamp,
                'robots_allowed': robots_allowed,
                'statut': 'collecte',
                'fichier_html_source': str(html_path.relative_to(PROJECT_ROOT)),
                'sha256_html': sha256,
                'octets_html': len(response.content),
                'message': '',
            })

        except requests.RequestException as erreur:
            traces.append({
                'id_document': document_id,
                'url': url,
                'date_collecte_utc': timestamp,
                'robots_allowed': robots_allowed,
                'statut': 'erreur_http',
                'message': str(erreur),
            })

        if position < len(documents) - 1:
            time.sleep(delai_secondes)

    return pd.DataFrame(traces)


In [ ]:
# 7. Exécuter ou simuler l'acquisition

if EXECUTER_COLLECTE_REELLE:
    if not CONDITIONS_ACCES_VERIFIEES:
        raise PermissionError(
            'Définissez CONDITIONS_ACCES_VERIFIEES = True seulement après vérification documentée.'
        )

    metadata_acquisition = collecter_pages_html(
        documents=urls_a_collecter,
        sources_dir=SOURCES_DIR,
        user_agent=USER_AGENT,
        robot_agent=ROBOT_AGENT,
        timeout_seconds=TIMEOUT_SECONDS,
        delai_secondes=DELAI_ENTRE_REQUETES,
    )
else:
    # En démonstration locale, on construit la trace correspondant au fichier fictif.
    html_bytes = fixture_path.read_bytes()
    metadata_acquisition = pd.DataFrame([{
        'id_document': 'D0001',
        'url': 'https://example.org/document-fictif',
        'url_finale': 'https://example.org/document-fictif',
        'date_collecte_utc': datetime.now(UTC).isoformat(),
        'robots_allowed': None,
        'statut': 'demonstration_locale',
        'fichier_html_source': str(fixture_path.relative_to(PROJECT_ROOT)),
        'sha256_html': hashlib.sha256(html_bytes).hexdigest(),
        'octets_html': len(html_bytes),
        'message': 'Aucune requête réseau : fixture locale.',
    }])

metadata_acquisition_path = INTERMEDIATE_DIR / 'metadata_acquisition.csv'
metadata_acquisition.to_csv(metadata_acquisition_path, index=False, encoding='utf-8')
metadata_acquisition


## Extraction des métadonnées HTML

Cette étape lit les fichiers HTML déjà acquis. Elle extrait des balises `meta` courantes sans supposer qu’une seule convention est universelle.

Les métadonnées produites restent **intermédiaires**. Elles doivent être vérifiées avant d’être intégrées à l’inventaire final `donnees/inventaire.csv`.


In [ ]:
# 8. Extraire les métadonnées depuis les fichiers HTML sources

metadonnees_completees = []

for _, ligne in metadata_acquisition.iterrows():
    if ligne['statut'] not in {'collecte', 'demonstration_locale'}:
        continue

    source_path = PROJECT_ROOT / ligne['fichier_html_source']
    soup = BeautifulSoup(source_path.read_text(encoding='utf-8'), 'html.parser')

    auteurs = tous_meta(soup, 'citation_author', 'citation_authors', 'DC.creator')
    mots_cles_bruts = tous_meta(soup, 'citation_keywords', 'keywords', 'DC.subject')
    mots_cles = list(dict.fromkeys(
        mot.strip()
        for valeur in mots_cles_bruts
        for mot in re.split(r'[,;]', valeur)
        if mot.strip()
    ))

    metadata = {
        **ligne.to_dict(),
        'titre_meta': premier_meta(soup, 'citation_title', 'DC.title', 'og:title'),
        'type_document': premier_meta(soup, 'og:type', 'DC.type', 'citation_article_type'),
        'doi': premier_meta(soup, 'citation_doi', 'DOI', 'DC.identifier'),
        'langue': premier_meta(soup, 'citation_language', 'DC.language', 'og:locale'),
        'date_publication': premier_meta(soup, 'citation_publication_date', 'DC.date', 'article:published_time'),
        'journal': premier_meta(soup, 'citation_journal_title', 'DC.source'),
        'volume': premier_meta(soup, 'citation_volume'),
        'issue': premier_meta(soup, 'citation_issue'),
        'resume': premier_meta(soup, 'citation_abstract', 'DC.description', 'description'),
        'auteurs': json.dumps(auteurs, ensure_ascii=False),
        'mots_cles': json.dumps(mots_cles, ensure_ascii=False),
    }

    document_id = metadata['id_document']
    json_path = INTERMEDIATE_DIR / f'{document_id}_metadonnees.json'
    json_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    metadonnees_completees.append(metadata)

metadata_complete = pd.DataFrame(metadonnees_completees)
metadata_complete_path = INTERMEDIATE_DIR / 'metadata_complete.csv'
metadata_complete.to_csv(metadata_complete_path, index=False, encoding='utf-8')
metadata_complete


## Extraction d’un aperçu de texte HTML

Cette extraction sert à comprendre le rôle d’un sélecteur CSS. Elle ne remplace pas le protocole d’extraction final, qui devra être stabilisé dans :

```text
chaine/04_preparation/01_extraction/
```

Le sélecteur `#content, article, main` est volontairement général. Sur une source réelle, l’étudiant doit inspecter la page et adapter le sélecteur.


In [ ]:
# 9. Extraire un aperçu textuel depuis le HTML

manifest_extraction = []

for _, ligne in metadata_complete.iterrows():
    document_id = ligne['id_document']
    source_path = PROJECT_ROOT / ligne['fichier_html_source']
    soup = BeautifulSoup(source_path.read_text(encoding='utf-8'), 'html.parser')

    # Retirer uniquement des éléments clairement hors texte principal.
    for element in soup.select('script, style, noscript, svg'):
        element.decompose()

    conteneur = soup.select_one(SELECTEUR_CONTENU)

    if conteneur is None:
        manifest_extraction.append({
            'id_document': document_id,
            'statut': 'selecteur_introuvable',
            'fichier_sortie': '',
        })
        continue

    paragraphes = [
        normaliser_espaces(paragraphe.get_text(' ', strip=True))
        for paragraphe in conteneur.select('p')
    ]

    texte = '

'.join(paragraphe for paragraphe in paragraphes if paragraphe)
    output_path = INTERMEDIATE_DIR / f'{document_id}_apercu_texte.txt'
    output_path.write_text(texte, encoding='utf-8')

    manifest_extraction.append({
        'id_document': document_id,
        'statut': 'apercu_extrait',
        'fichier_sortie': str(output_path.relative_to(PROJECT_ROOT)),
        'nombre_paragraphes': len(paragraphes),
        'nombre_caracteres': len(texte),
    })

manifest_html = pd.DataFrame(manifest_extraction)
manifest_html_path = INTERMEDIATE_DIR / 'extraction_html.csv'
manifest_html.to_csv(manifest_html_path, index=False, encoding='utf-8')
manifest_html


## Vérification de l’aperçu extrait

Le fichier extrait doit être comparé au HTML visible dans le navigateur ou au document source. Il faut notamment vérifier :

- les titres et sous-titres ;
- les paragraphes manquants ;
- les notes ;
- les références ;
- les caractères spéciaux ;
- les tableaux et figures ;
- les menus ou éléments résiduels ;
- la pertinence du sélecteur CSS.


In [ ]:
# 10. Lire l'aperçu textuel de démonstration

for _, ligne in manifest_html.iterrows():
    if ligne['statut'] == 'apercu_extrait':
        preview_path = PROJECT_ROOT / ligne['fichier_sortie']
        print(f'--- {ligne["id_document"]} ---')
        print(preview_path.read_text(encoding='utf-8')[:500])


## Extraction PDF : étape séparée

Le notebook ne télécharge pas automatiquement de PDF. Si un PDF a été acquis de manière autorisée et documentée, l’extraction peut être réalisée à partir d’une liste explicite :

```csv
id_document,pdf_path
D0001,donnees/sources/D0001_source.pdf
```

Cette séparation évite d’associer automatiquement un PDF au mauvais document ou de confondre le lien PDF repéré avec une autorisation de téléchargement et de redistribution.


In [ ]:
# 11. Fonction facultative d'extraction PDF
#
# Cette cellule ne s'exécute que si un manifest PDF explicite existe.

from pypdf import PdfReader

pdf_manifest_path = CONFIG_DIR / 'pdf_a_extraire.csv'

if pdf_manifest_path.exists():
    pdf_manifest = pd.read_csv(pdf_manifest_path, encoding='utf-8')
    sorties_pdf = []

    for _, ligne in pdf_manifest.iterrows():
        document_id = ligne['id_document']
        pdf_path = PROJECT_ROOT / ligne['pdf_path']

        try:
            reader = PdfReader(pdf_path)
            texte_pdf = '

'.join(page.extract_text() or '' for page in reader.pages)
            output_path = INTERMEDIATE_DIR / f'{document_id}_pdf_extrait.txt'
            output_path.write_text(texte_pdf, encoding='utf-8')
            sorties_pdf.append({
                'id_document': document_id,
                'statut': 'extrait',
                'pages': len(reader.pages),
                'fichier_sortie': str(output_path.relative_to(PROJECT_ROOT)),
            })
        except Exception as erreur:
            sorties_pdf.append({
                'id_document': document_id,
                'statut': 'erreur',
                'pages': None,
                'fichier_sortie': '',
                'message': str(erreur),
            })

    pd.DataFrame(sorties_pdf).to_csv(
        INTERMEDIATE_DIR / 'extraction_pdf.csv',
        index=False,
        encoding='utf-8',
    )
else:
    print('Aucun fichier configuration/pdf_a_extraire.csv : extraction PDF non exécutée.')


## Selenium facultatif : voir le navigateur et le DOM

Selenium est utile pour observer une page dynamique, inspecter le DOM et tester un sélecteur CSS. Il n’est pas nécessaire pour l’acquisition HTTP contrôlée ci-dessus.

La cellule suivante est désactivée par défaut. Pour l’exécuter, installer Selenium, mettre `EXECUTER_SELENIUM = True`, puis vérifier les conditions d’accès de la page consultée.


In [ ]:
# 12. Inspection Selenium facultative
EXECUTER_SELENIUM = False

if EXECUTER_SELENIUM:
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support import expected_conditions as EC
        from selenium.webdriver.support.ui import WebDriverWait

        options = Options()
        options.add_argument(f'--user-agent={USER_AGENT}')
        options.add_argument('--window-size=1440,1000')

        driver = webdriver.Chrome(options=options)
        driver.get(URL_EXEMPLE_OPENEDITION)

        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))

        print('Titre :', driver.title)
        print('URL finale :', driver.current_url)
        print('Inspecter maintenant la page avec F12, puis fermer Chrome manuellement.')

    except ImportError:
        print('Selenium n’est pas installé : python -m pip install "selenium>=4.11,<5"')
else:
    print('Inspection Selenium désactivée.')


## Sauvegarde et inspection des fichiers produits

La structure de données principale est conservée en formats ouverts :

- CSV pour les tableaux ;
- JSON pour les métadonnées par document ;
- HTML pour les sources Web ;
- TXT pour les aperçus ou extractions intermédiaires.

Le notebook ne sauvegarde pas un objet binaire équivalent à un fichier `.RDS` ou `.pkl` comme source primaire de vérité. Ces formats peuvent être utilisés localement, mais les fichiers CSV et JSON restent plus faciles à examiner, versionner et documenter.


In [ ]:
# 13. Vérification finale des sorties de démonstration
fichiers_attendus = [
    metadata_acquisition_path,
    metadata_complete_path,
    manifest_html_path,
    INTERMEDIATE_DIR / 'D0001_metadonnees.json',
    INTERMEDIATE_DIR / 'D0001_apercu_texte.txt',
]

for fichier in fichiers_attendus:
    print(f'{fichier.relative_to(PROJECT_ROOT)} :', fichier.exists())

print('
Étape suivante : documenter les décisions dans journal/journal_de_bord.md,')
print('puis stabiliser l’extraction et le nettoyage dans chaine/04_preparation/.')


# Exercices

## Exercice 1 — Inspecter une source réelle

1. Ouvrir dans un navigateur :

   ```text
   https://journals.openedition.org/rac/263
   ```

2. Utiliser `F12` ou **Inspecter**.
3. Repérer un lien vers un numéro ou un article.
4. Identifier l’attribut `href`.
5. Proposer un sélecteur CSS ou XPath permettant de retrouver ce lien.
6. Ne pas lancer de collecte automatique de l’ensemble de la revue.

Documenter votre observation dans le journal de recherche.

## Exercice 2 — Créer une liste finie de documents candidats

Créer :

```text
configuration/urls_a_collecter.csv
```

avec trois à cinq URL explicitement choisies et justifiées selon la question de recherche.

Avant toute collecte réelle, préciser pour chaque URL :

- l’identifiant de document ;
- le genre ;
- la source ;
- le lien avec la question ;
- le statut d’accès ;
- les conditions de réutilisation à vérifier.

## Exercice 3 — Adapter un sélecteur CSS

À partir d’un HTML source acquis ou de la fixture locale :

1. tester le sélecteur `#content, article, main` ;
2. comparer l’aperçu textuel avec la page source ;
3. choisir un sélecteur plus précis si nécessaire ;
4. expliquer dans le journal pourquoi le sélecteur retenu est adéquat ;
5. signaler les éléments exclus ou perdus.

## Exercice 4 — Vérifier les métadonnées

À partir de `metadata_complete.csv` :

1. vérifier manuellement le titre, la date et la langue d’au moins un document ;
2. comparer les métadonnées extraites à la page source ;
3. consigner les erreurs ou informations absentes ;
4. décider quelles métadonnées devront figurer dans `donnees/inventaire.csv`.


## Liens avec le dépôt Git

| Étape | Fichiers concernés |
|---|---|
| Question et cadre | `chaine/01_question_et_cadre/` |
| Conception du corpus | `chaine/02_conception_du_corpus/` |
| Acquisition | `chaine/03_acquisition/` |
| Préparation | `chaine/04_preparation/` |
| Données et inventaire | `donnees/` |
| Décisions et difficultés | `journal/` |
| Résultats finaux | `resultats/` |
| Méthodes de l’article | `article/article_imrad.md` |

Les données sources, fichiers intermédiaires et journaux techniques sont généralement exclus avec `.gitignore` lorsqu’ils sont volumineux, non redistribuables ou sous droits. Les scripts, protocoles, fichiers de configuration, inventaires et README doivent en revanche rester versionnés.
